In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 95 samples and 5850 columns.
First few rows of the data:


,time to_bcr_(days),biochemical recurrence_within_5_years_of_treatment,sample_title,tissue,treatment,previous treatment,age at_diagnosis,bmi,erg status,gleason grade_group_at_rp,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,NaN,0.0,3T,Prostate tumor - FFPE,Radical prostatectomy,No,64,26.49,high,G2_Gl7,...,5.653072,3.735675,5.754922,6.377859,4.443601,4.608845,6.780463,6.424128,4.432975,6.144973
1,119.0,1.0,5T,Prostate tumor - FFPE,Radical prostatectomy,No,55,24.62,low,G2_Gl7,...,7.115075,4.565450,6.862079,7.626646,5.357571,5.985145,8.023216,7.360952,5.458961,7.142796
2,1577.0,1.0,6T,Prostate tumor - FFPE,Radical prostatectomy,No,72,23.32,low,G2_Gl7,...,7.177151,4.577825,7.058581,7.599866,5.703526,5.980037,7.990661,7.282877,5.285521,7.433937
3,NaN,0.0,8T,Prostate tumor - FFPE,Radical prostatectomy,No,65,28.65,high,G3_Gl7,...,5.979267,3.950907,6.280321,7.039318,4.867194,4.847216,7.206353,6.636354,4.596222,6.516484
4,NaN,0.0,16T,Prostate tumor - FFPE,Radical prostatectomy,No,59,21.87,high,G4_5_higherGl7,...,6.436773,4.204412,6.245299,6.783016,4.872407,5.230882,7.154715,6.918546,5.172573,6.348771


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for time to_bcr_(days) or biochemical recurrence_within_5_years_of_treatment: 52
Number of samples with complete data for both variables: 43
Number of censored samples (event_var == 0.0): 0
Number of events (event_var == 1.0): 43
Censored ratio (among complete): 0 / 43 = 0.000 (0.0%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE216490/description.json
